In [2]:
import sys
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
sys.path.append("../src") 
from data.dataset import RAFCETrainDataset, RAFCEValDataset, get_weighted_sampler
from models.train_utils import train_model

In [3]:
train_df = pd.read_csv("../data/train_split.csv")
val_df = pd.read_csv("../data/val_split.csv")

In [4]:
train_ds = RAFCETrainDataset(train_df, img_dir="../data/train_images")
val_ds = RAFCEValDataset(val_df, img_dir="../data/val_images")
sampler = get_weighted_sampler(train_df)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

In [5]:
print("Loading ViT-B/16...")
model_vit = models.vit_b_16(weights='IMAGENET1K_V1')

Loading ViT-B/16...


In [6]:
in_features = model_vit.heads.head.in_features
model_vit.heads.head = nn.Linear(in_features, 14)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

Using device: cpu


In [8]:
optimizer_vit = optim.AdamW(model_vit.parameters(), lr=5e-5, weight_decay=0.01)
model_vit, history_vit = train_model(
    model_vit, train_loader, val_loader, 
    criterion, optimizer_vit, 
    num_epochs=15, device=device,
    save_path='best_vit_model.pth'
)

Epoch 1/15
----------


KeyboardInterrupt: 